In [ ]:
"""
This code will be my attempt in fully building and implementing a PINN in order to model non-linear partial differential equations such as the advection-diffusion
equation and Burger's equation.
"""

In [1]:
# Need to overload some operators in order to do algebra!

class Vector:
    def __init__(self, *components):
        """
        Create a vector of arbitrary length.
        Example:
            Vector(1, 2)
            Vector(1, 2, 3, 4)
        """
        if len(components) == 0:
            self.components = (0,)
        else:
            if not all(isinstance(x, (int, float)) for x in components):
                raise TypeError("All components must be numbers")
            self.components = tuple(components)

    def __repr__(self):
        return f"Vector{self.components}"

    def __len__(self):
        return len(self.components)

    def __getitem__(self, index):
        return self.components[index]

    # Vector addition
    def __add__(self, other):
        if not isinstance(other, Vector):
            return NotImplemented
        
        if len(self) != len(other):
            raise ValueError("Vectors must have same dimension")

        return Vector(*(a + b for a, b in zip(self.components, other.components)))
    
    # Scalar multiplication
    def __mul__(self, scalar):
        if not isinstance(scalar, (int, float)):
            return NotImplemented
        
        return Vector(*(scalar * x for x in self.components))

    # Allow scalar * vector
    def __rmul__(self, scalar):
        return self.__mul__(scalar)

    # Dot product using @
    def __matmul__(self, other):
        if not isinstance(other, Vector):
            return NotImplemented
        
        if len(self) != len(other):
            raise ValueError("Vectors must have same dimension")

        return sum(a * b for a, b in zip(self.components, other.components))

    # Magnitude
    def norm(self):
        return sum(x**2 for x in self.components) ** 0.5
    
# Tests
v1 = Vector(1, 2)
v2 = Vector(3, 4)

v3 = Vector(1, 2, 3)
v4 = Vector(4, 5, 6)

v5 = Vector(1, 2, 3, 4)
v6 = Vector(5, 6, 7, 8)

print(v1)
print(v2)

print(v3)
print(v4)

print(v1 + v2)
print(v1 + -1*v2)
print(v3 @ v4)
print(v3.norm())

Vector(1, 2)
Vector(3, 4)
Vector(1, 2, 3)
Vector(4, 5, 6)
Vector(4, 6)
Vector(-2, -2)
32
3.7416573867739413


In [5]:
class HyperDual:
    # Need to make a HyperDual class separate from vectors so that I can also make a new algebra.
    # I had previously tried to do it in 1 class with 16 components and that was very stupid.

    def __init__(self, real, eps1=0, eps2=0, eps12=0):

        self.real = real
        self.eps1 = eps1
        self.eps2 = eps2
        self.eps12 = eps12

    def __add__(self, other):

        return HyperDual(self.real + other.real, self.eps1 + other.eps1, self.eps2 + other.eps2, self.eps12 + other.eps12)

    def __mul__(self, other):
        # HyperDual multiplication
        if isinstance(other, HyperDual):

            a, b, c, d = (self.real, self.eps1, self.eps2, self.eps12)

            e, f, g, h = (other.real, other.eps1, other.eps2, other.eps12)

            return HyperDual(a * e, a * f + b * e, a * g + c * e, a * h + b * g + c * f + d * e)
    
        # Scalar multiplication (useful for subtraction)
        elif isinstance(other, (int, float)):

            return HyperDual(other * self.real, other * self.eps1, other * self.eps2, other * self.eps12)

        return NotImplemented
    
    def __rmul__(self, other):
        return self.__mul__(other)

    def hyp_div(self): 

        if len(self) != 4:
            raise ValueError("Hyper-dual vectors must have dimension 4")
        
        a, b, c, d = (self.real, self.eps1, self.eps2, self.eps12)

        if a == 0:
            raise ZeroDivisionError("Real component cannot be zero")

        return HyperDual(1/a, -b/a**2, -c/a**2, (2*b*c / a**3) - (d / a**2))
    
    # Magnitude of a hyper dual vector is given by the real component.
    def norm(self):

        return abs(self.real)
    
    def scalarmult(self, scalar):
        if not isinstance(scalar, (int, float)):
            return NotImplemented
        
        return HyperDual(*(scalar * x for x in (self.real, self.eps1, self.eps2, self.eps12)))
